In [76]:
import pandas as pd
import numpy as np
import joblib

laptime_model = joblib.load('lap_time.joblib')
model_columns = joblib.load('lap_model_columns.joblib')

In [77]:
track_df = pd.read_csv('track_char.csv')
track_df['EventName'] = track_df['EventName'].str.strip()
track_df['TrackDirection'] = track_df['TrackDirection'].str.strip().str.capitalize()

Target_event = 'Bahrain Grand Prix'
track_row = track_df[track_df['EventName'] == Target_event].iloc[0]


In [78]:
sim_weather = {'TrackTemp': 35.0, 'Airtemp': 25.0}
sim_driver = 'VER'
Race_laps = 57

In [79]:
def build_feature_row(compound, tyre_life, race_lap_number, driver, track_row, weather, model_columns):
    raw_row = {
        'TyreLife': tyre_life,
        'FreshTyre': True if tyre_life <= 1 else False,
        'Stint': 1,
        'TrackTemp': weather['TrackTemp'],
        'AirTemp': weather['Airtemp'],
        'LapNumber': race_lap_number, 
        'CircuitLength_km': track_row['CircuitLength_km'],
        'NumCorners': track_row['NumCorners'],
        'NumDRSZones': track_row['NumDRSZones'],
        'AvgSpeed_kmh': track_row['AvgSpeed_kmh'],
        'ElevationChange_m': track_row['ElevationChange_m'],
    }
    
    df_row = pd.DataFrame([raw_row])
    df_row['Compound'] = compound
    df_row['Driver'] = driver
    df_row['TrackDirection'] = track_row['TrackDirection']
    df_row['DownforceLevel'] = track_row['DownforceLevel']
    
    df_row = pd.get_dummies(df_row, columns=['Compound', 'Driver', 'TrackDirection', 'DownforceLevel'])
    df_row = df_row.reindex(columns=model_columns, fill_value=0)
    
    return df_row

In [84]:
print(sim_weather)
print(type(sim_weather))

{'TrackTemp': 35.0, 'Airtemp': 25.0}
<class 'dict'>


In [83]:
test_row = build_feature_row('MEDIUM', 5, 1, sim_driver, track_row, sim_weather, model_columns)
print(test_row.shape, "should match:", len(model_columns))
print(laptime_model.predict(test_row))

(1, 49) should match: 49
[104.36422]


In [93]:
for tl in [1, 10, 20, 30, 40, 50]:
    row = build_feature_row('MEDIUM', tl, 57, sim_driver, track_row, sim_weather, model_columns)
    print(f"Predicted lap time for tyre life {tl}: {laptime_model.predict(row)[0]:.3f} seconds")

Predicted lap time for tyre life 1: 99.788 seconds
Predicted lap time for tyre life 10: 98.330 seconds
Predicted lap time for tyre life 20: 98.423 seconds
Predicted lap time for tyre life 30: 98.980 seconds
Predicted lap time for tyre life 40: 98.982 seconds
Predicted lap time for tyre life 50: 98.957 seconds


In [86]:
def simulate_stint(compound, start_tyre_life, start_race_lap, n_laps, driver, track_row, weather, model_columns):
    total_time = 0
    lap_times = []
    for lap_offset in range(n_laps):
        tyre_life = start_tyre_life + lap_offset
        race_lap = start_race_lap + lap_offset
        row = build_feature_row(compound, tyre_life, race_lap, driver, track_row, weather, model_columns)
        predicted_time = laptime_model.predict(row)[0]
        lap_times.append(predicted_time)
        total_time += predicted_time
    return total_time, lap_times

In [87]:
PIT_LOSS_SECONDS = 22

def evaluate_strategy(strategy, race_laps, driver, track_row, weather, model_columns):
    total_time = 0
    laps_remaining = race_laps
    current_race_lap = 1
    n_stints = len(strategy['stints'])
    stint_breakdown = []

    for i, (compound, stint_length) in enumerate(strategy['stints']):
        if stint_length is None:
            stint_length = laps_remaining // (n_stints - i)
        stint_time, lap_times = simulate_stint(compound, 1, current_race_lap, stint_length, 
                                                 driver, track_row, weather, model_columns)
        total_time += stint_time
        stint_breakdown.append({'compound': compound, 'laps': stint_length, 'stint_time': stint_time})
        laps_remaining -= stint_length
        current_race_lap += stint_length
        if i < n_stints - 1:
            total_time += PIT_LOSS_SECONDS

    return total_time, stint_breakdown

In [88]:
CANDIDATE_STRATEGIES = [
    {"name": "1-Stop: Medium-Hard", "stints": [("MEDIUM", None), ("HARD", None)]},
    {"name": "1-Stop: Hard-Medium", "stints": [("HARD", None), ("MEDIUM", None)]},
    {"name": "2-Stop: Medium-Medium-Hard", "stints": [("MEDIUM", None), ("MEDIUM", None), ("HARD", None)]},
    {"name": "2-Stop: Soft-Medium-Hard", "stints": [("SOFT", None), ("MEDIUM", None), ("HARD", None)]},
]

results = []
for strategy in CANDIDATE_STRATEGIES:
    total_time, breakdown = evaluate_strategy(strategy, Race_laps, sim_driver, track_row, sim_weather, model_columns)
    results.append({'strategy': strategy['name'], 'predicted_total_time_sec': total_time})

results_df = pd.DataFrame(results).sort_values('predicted_total_time_sec')
print(results_df)

                     strategy  predicted_total_time_sec
1         1-Stop: Hard-Medium               5701.208510
3    2-Stop: Soft-Medium-Hard               5707.550506
2  2-Stop: Medium-Medium-Hard               5708.743709
0         1-Stop: Medium-Hard               5713.752709


In [89]:
breakdown = evaluate_strategy(CANDIDATE_STRATEGIES[2], Race_laps, sim_driver, track_row, sim_weather, model_columns)
print(breakdown)

(np.float64(5708.743709333335), [{'compound': 'MEDIUM', 'laps': 19, 'stint_time': np.float64(1866.3602593333337)}, {'compound': 'MEDIUM', 'laps': 19, 'stint_time': np.float64(1868.6240200000007)}, {'compound': 'HARD', 'laps': 19, 'stint_time': np.float64(1929.7594299999998)}])
